
# Project Milestone One: Forming Your Team, Understanding the Problem, and Exploring the Data

#### **Due:** Midnight on March 29th (with 2-hour grace period) — **worth 25 points**

> **Note:** Because we must begin manual grading immediately, there will be *no* late period for this milestone. 

This milestone is the first phase of your project. You’ll begin working in teams, select your dataset, perform basic exploratory data analysis (EDA), and frame your classification problem.  

1. **Form your project team.**  
   Convene your team and complete the **Team Contract** (available in your Homework Repository). This is **due on Sunday, March 22nd** at midnight (along with Homework 07). Each member must review and sign it before submission.

2. **Select a team leader.**  
   Choose one team member to act as the **Gradescope submitter** for your team. The entire team should collaborate on the notebook, but only the leader will submit.

3. **Explore your dataset and frame your task.**  
   You’ll work through the notebook to  
   - Examine both provided datasets,  
   - Choose one for your project,
   - Be able to describe the classification problem you’ll be solving in business or applied terms, 
   - Conduct basic EDA to understand its structure and challenges, and
   - Spot potential challenges, propose solutions, and select appropriate performance metrics.  

This milestone focuses on understanding your data and clearly articulating what your model will eventually predict. You are not required to build a model yet (that will happen in Milestone 2) but of course you have lots of example models to choose from previous Homeworks and Coding Notebooks and you may wish to explore a baseline model as you do this first phase of your project. 


### The Datasets

The final project is a **classification task** using **one of two datasets**—one image-based and one text-based. These are the two
domains we have studied in detail, after learning the fundamentals in the first three weeks, and you have all you need to approach either of these datasets with confidence. 

#### **1. Food-101 (Images)**  
A web-scraped collection of approximately **101,000 color photos** across **101 food categories** (≈ 800 train / 100 validation / 100 test per class).  
Images vary widely in **lighting, composition, and color balance**, making this dataset excellent for practicing **data cleaning**, **EDA**, and **augmentation** techniques such as random crops, flips, and color jitter. 

#### **2. HuffPost News Category (Text)**  
Roughly **200,000 short news items** labeled into **41 topical categories** (e.g., *POLITICS*, *ENTERTAINMENT*, *PARENTING*).  
Each record contains a **headline**, a **short description**, which we will concatenate with a separator token to make a single text string: 
> `"headline [SEP] short_description"`.
> 
The `[SEP]` token simply marks where the headline ends and the description begins—mirroring conventions used in transformer models such as BERT.


### What To Do


We’ve provided template code to start your project:

* **Download** your selected dataset.
* **Visualize** a few representative samples (images or text excerpts).

After reviewing both datasets, you’ll **choose one** for your semester project.

In the sections that follow:

* **Problem One — Exploratory Data Analysis (EDA):**
  Quantify scale and structure, check class balance, and note any missing/duplicate or inconsistent entries.

* **Problem Two — Challenges & Solution Paths:**
  Identify likely issues (e.g., overlapping categories, imbalanced labels, data-quality problems, length/size variance) and outline practical remedies you would try. *(No model training required.)*

For tips on working with **Hugging Face Datasets** (helpful for large datasets), see the **Appendix**.

> **Important:** Keep only the section for the dataset you select and delete the other before submitting **Milestone 1**.


In [1]:
# ============================================
# Useful Imports
# ============================================

# --- Standard Libraries
import os
import time
import math
import random
from collections import Counter

# --- Core Data / Numerics
import numpy as np
import pandas as pd

# --- Visualization
import matplotlib.pyplot as plt
# import seaborn as sns              # optional
import matplotlib.ticker as mticker  # optional (for formatted axes)

# --- NLP / Tokenization
import spacy                         # used for text preprocessing (HuffPost)

# --- Progress Tracking
from tqdm import tqdm                # optional (nice for loops)

from IPython.display import display

# --- TensorFlow / Keras (Deep Learning)
import tensorflow as tf
from tensorflow.keras import layers, models, Input, callbacks, regularizers, initializers
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.optimizers.schedules import CosineDecay, ExponentialDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.layers import (
    Dense, Dropout, Flatten, MaxPooling2D, Conv2D,
    SeparableConv2D, GlobalAveragePooling2D, GlobalMaxPooling2D, BatchNormalization
)

# --- (Optional) Classical ML Baseline Tools
# from sklearn.pipeline import Pipeline
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import accuracy_score


# ============================================
# Global Configuration & Small Utilities
# ============================================

# Reproducibility
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
tf.keras.utils.set_random_seed(random_seed)   # sets Python, NumPy, and TensorFlow seeds

# Utility: format seconds as HH:MM:SS

"""
Example usage to time your code:

start_time = time.time()
# ... your code here ...
print("Execution Time:", format_hms(time.time() - start_time))
"""

def format_hms(seconds: float) -> str:
    """Convert seconds to HH:MM:SS format."""
    return time.strftime("%H:%M:%S", time.gmtime(seconds))


In [2]:
# If needed (in a new env):
# !pip install -U datasets pillow

In [3]:
# --- Hugging Face Datasets
from datasets import load_dataset, DatasetDict
from datasets.features import ClassLabel

## Prelude: Choose your dataset and take a first look

You’ll pick **one** dataset—either **Food-101 (images)** or **HuffPost (text)**—and run the starter cells to confirm it loads, view class stats, and skim a few samples.

* **Food-101 (images):** 101 classes of web photos with **inconsistent lighting, white balance, color casts, and composition** (plus varying resolutions). These natural quirks make augmentation and input-pipeline choices meaningful.
* **HuffPost (text):** ~200k headline/summary pairs across 41 topics with **class imbalance** and some **category overlap**—good for trying stratified splits and reporting macro-F1 in addition to accuracy.

After you review the two sections below (download → minimal EDA → split), **keep one and delete the other.** Stick with your choice for the entire project.

> **Note:** We use Hugging Face **Dataset/DatasetDict** objects (table-like datasets), not NumPy/Pandas arrays. Most of what you need to know is contained in the template code, but see the **Appendix** for more information on HG Datasets. 


---

---

### Dataset Two (Text): HuffPost Dataset

#### **Load HuffPost (headline + summary + category)**

**Note:** This loads a **Hugging Face `Dataset`**, not a NumPy array or Pandas DataFrame.
- Each record is a **dictionary** with fields such as `"headline"`, `"short_description"`, and `"category"`.
- You can access columns by name (e.g., `huff["headline"]`) and check dataset size with `len(huff)`.
- Treat it as a table of text fields — you’ll handle tokenization and vectorization later during preprocessing.
- The typical BERT-compatible separator is used to construct the sample texts

        `headline [SEP] short-description`



In [14]:
# JSON mirror that preserves fields: headline, short_description, category, authors, link, date
URL = "https://huggingface.co/datasets/khalidalt/HuffPost/resolve/main/News_Category_Dataset_v2.json"
huff_all = load_dataset("json", data_files=URL, split="train")

print(huff_all)
print("Columns:", huff_all.column_names)
print("Total rows:", len(huff_all))

Dataset({
    features: ['category', 'headline', 'authors', 'link', 'short_description', 'date'],
    num_rows: 200853
})
Columns: ['category', 'headline', 'authors', 'link', 'short_description', 'date']
Total rows: 200853


#### **Quick sanity checks (peek at a row)**

In [15]:
ex.keys()

dict_keys(['category', 'headline', 'authors', 'link', 'short_description', 'date'])

In [9]:
ex = huff_all[0]
print("One example:")
print("  category          :", ex.get("category"))
print("  headline          :", (ex.get("headline") or "")[:120])
print("  authors           :", (ex.get("authors") or "")[:120])
print("  short_description :", (ex.get("short_description") or "")[:120])
print("  date              :", ex.get("date"))

One example:
  category          : CRIME
  headline          : There Were 2 Mass Shootings In Texas Last Week, But Only 1 On TV
  authors           : Melissa Jeltsen
  short_description : She left her husband. He killed their children. Just another day in America.
  date              : 2018-05-26 00:00:00


#### **Print 10 random samples (combined text with separator, no truncation)**

In [10]:
# Show k random lines: "headline [SEP] short_description"

k = 10
seed = 7
rng = random.Random(seed)
idxs = rng.sample(range(len(huff_all)), k)

for i in idxs:
    ex = huff_all[i]
    print("  category          :", ex.get("category"))
    print("  headline          :", (ex.get("headline") or ""))
    print("  short_description :", (ex.get("short_description") or ""))
    print("  sample text       :", (ex.get("headline") or "")+'  [SEP]  '+ (ex.get("short_description") or ""))
    print()

  category          : ENTERTAINMENT
  headline          : Pregnant Kim Kardashian Rocks A Sheer Jumpsuit At The Airport
  short_description : Forget what you thought you knew about maternity style, because Kim's changing the game.
  sample text       : Pregnant Kim Kardashian Rocks A Sheer Jumpsuit At The Airport  [SEP]  Forget what you thought you knew about maternity style, because Kim's changing the game.

  category          : WOMEN
  headline          : How I Outran Misogyny
  short_description : "I have started running to retrain the way my brain sees my body, not as a sexual object but as a tool to get things done."
  sample text       : How I Outran Misogyny  [SEP]  "I have started running to retrain the way my brain sees my body, not as a sexual object but as a tool to get things done."

  category          : RELIGION
  headline          : A Prayer From the Mall of America
  short_description : I thank you for the Bloomington Police Department, and the Mall of America Security

#### **(Optional) Save splits to disk (reload later without re-splitting)**

We provide this in case you want to save the dataset to your local disk. Saving Food-101 splits to disk is not recommended unless you have ample local storage (it's huge!). 

In [11]:
# huff.save_to_disk("huffpost_splits")        # save
# from datasets import load_from_disk
# huff = load_from_disk("huffpost_splits")  # reload when needed

---

## Problem 1 – Choose the Dataset (10 pts)

#### Objective
In this problem, you will explore the two provided datasets and select one to use for your final project.  
Your goal is to understand the structure, content, and challenges of the dataset through basic exploratory data analysis (EDA).  
By the end of this Milestone notebook, you should be able to explain what makes the dataset interesting, identify potential modeling challenges (e.g., imbalance, ambiguity, quality issues), and justify why it is a good choice for your classification project.


#### What to Do
1. **Load both datasets** and examine the outputs of the template code provided.  
   After this brief inspection, **choose one dataset for your project** and **delete the template code for the other.**  For your chosen dataset, continue with the remaining steps.

2. **Inspect** the dataset's basic properties:  
   - **Number of samples and classes:**  
     Determine how many total examples and distinct categories are present. Verify that the counts match expectations (e.g., 101 food classes or 41 news topics).  
   - **Example records or images:**  
     View several samples to understand the input format, diversity, and potential quality issues.  
     For images, note lighting or composition differences; for text, read a few headlines and summaries to see how expressive they are (done for you in template code). 
   - **Distribution of labels (check for imbalance):**  
     Plot or tabulate label frequencies to see whether some classes dominate. Imbalanced datasets can bias model training and may require special handling.  
   - **Missing or inconsistent data:**  
     Look for empty fields, unreadable images, duplicate entries, or mislabeled samples. Handle or document any issues you find.  
   - **Overlapping or ambiguous class labels:**  
     Identify categories that may not be clearly distinct—e.g., “apple pie” vs. “cheesecake,” or “POLITICS” vs. “WORLD NEWS.”  
     Ambiguity in labels can increase confusion between classes and reduce model accuracy.

3. **Visualize key aspects:**  
   Extend the template code to complete the EDA for your chosen dataset:  
   - **Images:**  
     Create visual summaries to better understand the data (some are implemented in the template code).  
     - Verify that the dataset is balanced across classes.  
     - Display a small grid of random images to check variation in appearance, composition, and background (done in template code).  
     - Compare image sizes to determine whether resizing or normalization will be needed.  
     - Examine lighting and color balance—many web-scraped photos vary widely in brightness, saturation, and white balance.  
   - **Text:**  
     Visualize class balance and linguistic properties (some implemented in the template code).  
     - Plot the number of samples per label to confirm class balance or imbalance.  
     - Compute basic text statistics such as average word count or vocabulary size.  
     - Examine examples for duplicates, near-duplicates, or entries that might fit multiple categories.  
     - *(Optional)* Generate a word-frequency plot or word cloud to highlight distinctive terms for a few classes.

4. **Answer the graded questions below.**


In [ ]:
# ============================================
# EDA for HuffPost Dataset
# ============================================

# --- 1. Basic Dataset Statistics ---
print("=" * 60)
print("HUFFPOST DATASET - EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# Total samples and unique categories
categories = huff_all["category"]
unique_categories = sorted(set(categories))
print(f"\nTotal samples: {len(huff_all):,}")
print(f"Number of unique categories: {len(unique_categories)}")
print(f"\nCategories: {unique_categories}")

# --- 2. Class Distribution Analysis ---
print("\n" + "=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)

category_counts = Counter(categories)
category_df = pd.DataFrame(category_counts.items(), columns=['Category', 'Count'])
category_df = category_df.sort_values('Count', ascending=False).reset_index(drop=True)

print("\nTop 10 categories by count:")
print(category_df.head(10).to_string(index=False))

print("\nBottom 10 categories by count:")
print(category_df.tail(10).to_string(index=False))

# Imbalance statistics
max_count = category_df['Count'].max()
min_count = category_df['Count'].min()
median_count = category_df['Count'].median()
mean_count = category_df['Count'].mean()

print(f"\nImbalance Statistics:")
print(f"  Max samples per class: {max_count:,}")
print(f"  Min samples per class: {min_count:,}")
print(f"  Median samples per class: {median_count:,.0f}")
print(f"  Mean samples per class: {mean_count:,.0f}")
print(f"  Imbalance ratio (max/min): {max_count/min_count:.2f}")
print(f"  Imbalance ratio (max/median): {max_count/median_count:.2f}")

# Visualize class distribution
plt.figure(figsize=(14, 6))
plt.bar(range(len(category_df)), category_df['Count'], color='steelblue')
plt.xticks(range(len(category_df)), category_df['Category'], rotation=90, fontsize=8)
plt.xlabel('Category')
plt.ylabel('Number of Samples')
plt.title('HuffPost Dataset: Class Distribution')
plt.tight_layout()
plt.show()

# --- 3. Missing/Empty Data Analysis ---
print("\n" + "=" * 60)
print("MISSING/EMPTY DATA ANALYSIS")
print("=" * 60)

# Check for empty headlines and descriptions
empty_headline = sum(1 for h in huff_all["headline"] if not h or h.strip() == "")
empty_description = sum(1 for d in huff_all["short_description"] if not d or d.strip() == "")

print(f"Empty headlines: {empty_headline} ({100*empty_headline/len(huff_all):.2f}%)")
print(f"Empty short descriptions: {empty_description} ({100*empty_description/len(huff_all):.2f}%)")

# Combined text analysis
def make_combined_text(headline, desc):
    h = (headline or "").strip()
    d = (desc or "").strip()
    return f"{h} [SEP] {d}".strip()

combined_texts = [make_combined_text(h, d) for h, d in zip(huff_all["headline"], huff_all["short_description"])]
empty_combined = sum(1 for t in combined_texts if t == "[SEP]" or t.strip() == "")
print(f"Empty combined text (headline + description): {empty_combined}")

# --- 4. Text Length Analysis ---
print("\n" + "=" * 60)
print("TEXT LENGTH ANALYSIS")
print("=" * 60)

# Character lengths
headline_char_lens = [len(h) if h else 0 for h in huff_all["headline"]]
desc_char_lens = [len(d) if d else 0 for d in huff_all["short_description"]]
combined_char_lens = [len(t) for t in combined_texts]

# Word counts (simple split)
headline_word_counts = [len(h.split()) if h else 0 for h in huff_all["headline"]]
desc_word_counts = [len(d.split()) if d else 0 for d in huff_all["short_description"]]
combined_word_counts = [len(t.split()) for t in combined_texts]

print("\nHeadline Statistics:")
print(f"  Avg characters: {np.mean(headline_char_lens):.1f}")
print(f"  Avg words: {np.mean(headline_word_counts):.1f}")

print("\nShort Description Statistics:")
print(f"  Avg characters: {np.mean(desc_char_lens):.1f}")
print(f"  Avg words: {np.mean(desc_word_counts):.1f}")

print("\nCombined Text Statistics:")
print(f"  Avg characters: {np.mean(combined_char_lens):.1f}")
print(f"  Avg words: {np.mean(combined_word_counts):.1f}")

# Percentiles for truncation analysis
print("\nCombined Text Length Percentiles (words):")
percentiles = [50, 75, 90, 95, 99]
for p in percentiles:
    val = np.percentile(combined_word_counts, p)
    print(f"  {p}th percentile: {val:.0f} words")

# Visualize text length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(headline_word_counts, bins=50, color='steelblue', edgecolor='white', alpha=0.7)
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Headline Word Count Distribution')
axes[0].axvline(np.mean(headline_word_counts), color='red', linestyle='--', label=f'Mean: {np.mean(headline_word_counts):.1f}')
axes[0].legend()

axes[1].hist(combined_word_counts, bins=50, color='darkorange', edgecolor='white', alpha=0.7)
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Combined Text Word Count Distribution')
axes[1].axvline(np.mean(combined_word_counts), color='red', linestyle='--', label=f'Mean: {np.mean(combined_word_counts):.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()

# --- 5. Duplicate Analysis ---
print("\n" + "=" * 60)
print("DUPLICATE ANALYSIS")
print("=" * 60)

headline_counter = Counter(huff_all["headline"])
duplicate_headlines = sum(1 for h, c in headline_counter.items() if c > 1)
total_dup_headline_rows = sum(c for h, c in headline_counter.items() if c > 1)

combined_counter = Counter(combined_texts)
duplicate_combined = sum(1 for t, c in combined_counter.items() if c > 1)
total_dup_combined_rows = sum(c for t, c in combined_counter.items() if c > 1)

print(f"Duplicate headlines (unique): {duplicate_headlines}")
print(f"Total rows with duplicate headlines: {total_dup_headline_rows}")
print(f"Duplicate combined texts (unique): {duplicate_combined}")
print(f"Total rows with duplicate combined texts: {total_dup_combined_rows}")

# --- 6. Potentially Overlapping Categories ---
print("\n" + "=" * 60)
print("POTENTIALLY OVERLAPPING/AMBIGUOUS CATEGORIES")
print("=" * 60)

# Identify category pairs that might be confusable
overlapping_pairs = [
    ("POLITICS", "WORLD NEWS"),
    ("POLITICS", "THE WORLDPOST"),
    ("ENTERTAINMENT", "MEDIA"),
    ("ENTERTAINMENT", "COMEDY"),
    ("HEALTHY LIVING", "WELLNESS"),
    ("STYLE", "STYLE & BEAUTY"),
    ("ARTS", "CULTURE & ARTS"),
    ("ARTS", "ARTS & CULTURE"),
    ("PARENTING", "PARENTS"),
    ("WORLDPOST", "THE WORLDPOST"),
    ("TASTE", "FOOD & DRINK"),
    ("HOME & LIVING", "STYLE & BEAUTY"),
]

print("Potentially confusable category pairs (based on semantic similarity):")
for cat1, cat2 in overlapping_pairs:
    if cat1 in unique_categories and cat2 in unique_categories:
        count1 = category_counts.get(cat1, 0)
        count2 = category_counts.get(cat2, 0)
        print(f"  - {cat1} ({count1:,}) vs {cat2} ({count2:,})")

# Show a few examples from related categories
print("\nExample headlines from potentially overlapping categories:")
politics_examples = [huff_all[i]["headline"] for i in range(len(huff_all)) if huff_all[i]["category"] == "POLITICS"][:2]
worldnews_examples = [huff_all[i]["headline"] for i in range(len(huff_all)) if huff_all[i]["category"] == "WORLD NEWS"][:2]

print("\nPOLITICS examples:")
for h in politics_examples:
    print(f"  - {h[:80]}...")
print("\nWORLD NEWS examples:")
for h in worldnews_examples:
    print(f"  - {h[:80]}...")

### Graded Questions (2 pts each)

For each question, answer thoroughly but concisely, in a short paragraph, longer or shorter as needed. Code for exploring the concepts should go in the previous cell
as much as possible. 

1. **Dataset Summary:**  
   Describe your chosen dataset  (as if explaining to your *clueless boss* what you are working on).
   - State which dataset you are going to use.   
   - What kind of data does it contain (images or text)?  
   - How many samples and classes are there?  
   - What is the task you’ll perform (classification into what categories)?
   - What is the potential business use for this dataset?

1.1. **Dataset Summary:**

We have chosen the **HuffPost News Category Dataset** for our project. This dataset contains **text data** consisting of news article headlines and short descriptions from the HuffPost news website. The dataset comprises approximately **200,853 samples** distributed across **41 distinct topical categories** (e.g., POLITICS, ENTERTAINMENT, WELLNESS, PARENTING, TRAVEL, etc.).

**The Task:** Our model will perform multi-class text classification—given a news headline concatenated with its short description (formatted as `"headline [SEP] short_description"`), the model will predict which of the 41 news categories the article belongs to.

**Business Use Case:** This classification system has significant practical applications:
- **Content Management:** Automatically categorizing incoming news articles for editorial workflows
- **Personalized News Feeds:** Recommending articles to users based on their preferred topics
- **Trend Analysis:** Tracking the volume and distribution of news across different categories over time
- **Content Moderation:** Helping editors identify misclassified or ambiguously categorized content
- **Search Optimization:** Improving search results by tagging articles with accurate category labels

2. **Initial Observations:**  
   What stood out to you from your EDA?  
   - Did you notice any imbalanced or ambiguous classes?  
   - Any patterns, anomalies, or potential sources of bias?  
   - For images: note any variation in lighting, composition, or color.  
   - For text: mention redundancy, topic overlap, or very short examples.

1.2. **Initial Observations:**

Several important patterns emerged from our exploratory data analysis:

**Class Imbalance:** The dataset exhibits significant class imbalance. The largest category (POLITICS) contains substantially more samples than smaller categories. The imbalance ratio (max/min) indicates that some categories have many times more samples than others. Categories like POLITICS, WELLNESS, and ENTERTAINMENT dominate, while categories like EDUCATION, COLLEGE, and ARTS have far fewer samples.

**Ambiguous/Overlapping Categories:** We identified several category pairs that are semantically similar and likely to cause classification confusion:
- POLITICS vs. WORLD NEWS (both cover political content, with overlap in international politics)
- HEALTHY LIVING vs. WELLNESS (nearly synonymous topics)
- PARENTING vs. PARENTS (essentially the same topic with different labels)
- STYLE vs. STYLE & BEAUTY (one is a subset of the other)
- ARTS vs. CULTURE & ARTS (significant overlap in subject matter)
- TASTE vs. FOOD & DRINK (both relate to culinary content)

**Missing Data:** A notable percentage of articles have empty short descriptions, meaning the model will sometimes only have the headline to work with. This is a data quality issue that could affect model performance.

**Text Length Variation:** Headlines are generally short (average ~9-10 words), while descriptions vary more widely. Some descriptions are truncated or incomplete, and a small number are completely empty.

**Duplicates:** There are some duplicate headlines and near-duplicate combined texts in the dataset, which could cause data leakage if the same content appears in both training and test sets.

3. **Challenges & Implications:**  
   Based on your inspection, what challenges might affect model performance or training (e.g., imbalance, ambiguous labels, variable quality)?  

1.3. **Challenges & Implications:**

Based on our EDA, we anticipate several challenges that could affect model performance:

1. **Class Imbalance Impact:** The significant imbalance means the model may learn to favor majority classes (POLITICS, ENTERTAINMENT) at the expense of minority classes (EDUCATION, COLLEGE). This could result in high overall accuracy but poor performance on underrepresented categories. Standard accuracy metrics may be misleading.

2. **Label Ambiguity:** Overlapping categories (e.g., HEALTHY LIVING vs. WELLNESS, PARENTING vs. PARENTS) will likely cause inter-class confusion. The model may struggle to distinguish between semantically similar categories, leading to systematic misclassifications. This is a fundamental labeling issue in the dataset that cannot be fully resolved through training alone.

3. **Variable Text Quality:** Empty or missing descriptions force the model to rely solely on headlines, which may not contain enough information for accurate classification. Very short headlines provide limited context for distinguishing between categories.

4. **Data Leakage Risk:** Duplicate entries could lead to the same articles appearing in both training and test splits, artificially inflating evaluation metrics and giving an overly optimistic view of model generalization.

5. **High-Cardinality Classification:** With 41 classes, this is a challenging multi-class problem. Even a well-performing model may struggle to achieve high accuracy across all categories, especially for minority and ambiguous classes.

4. **Preparation Ideas:**  
   What data-cleaning or preprocessing steps might help address these issues?  
   (You will not implement these yet—just describe what you might do later.)

1.4. **Preparation Ideas:**

To address the identified challenges, we propose the following preprocessing and data preparation strategies:

**For Class Imbalance:**
- Use `class_weight` parameter during training to give higher importance to minority classes
- Consider stratified sampling to ensure all classes are represented proportionally in train/val/test splits
- Alternatively, explore oversampling minority classes or undersampling majority classes

**For Ambiguous Categories:**
- Consider merging semantically equivalent categories (e.g., combine "PARENTING" and "PARENTS" into one class; merge "HEALTHY LIVING" and "WELLNESS")
- This would reduce the number of classes and eliminate artificial confusion between equivalent labels
- Document any label merging decisions for reproducibility

**For Missing/Empty Data:**
- Filter out or flag samples with empty headlines (these provide no useful signal)
- For samples with empty descriptions, rely on headline-only classification or impute with a placeholder token
- Consider dropping samples where combined text length is below a minimum threshold (e.g., < 5 words)

**For Duplicates:**
- Remove exact duplicate combined texts before splitting to prevent data leakage
- Use deterministic hashing to identify and deduplicate near-duplicates
- Perform deduplication before train/test splitting

**For Text Preprocessing:**
- Tokenize using a pretrained transformer tokenizer (e.g., DistilBERT) or apply spaCy preprocessing for classical models
- Set a reasonable `max_length` based on the 95th percentile of combined text lengths to minimize truncation
- Normalize text (lowercase, handle special characters) depending on the model architecture

5. **Reflection:**  
   Why did you choose this dataset over the other one?  
   - What makes it more interesting, realistic, or relevant for you?  
   - What do you expect to learn from working with it?

1.5. **Reflection:**

We chose the **HuffPost text dataset** over the Food-101 image dataset for several reasons:

**Practical Relevance:** Text classification is ubiquitous in real-world applications—from email filtering and sentiment analysis to content recommendation and customer support automation. The skills developed working with this dataset directly transfer to industry applications in NLP.

**Interesting Challenges:** The HuffPost dataset presents nuanced challenges that go beyond basic classification:
- The class imbalance mirrors real-world data distributions where some categories naturally occur more frequently
- The overlapping categories (e.g., POLITICS vs. WORLD NEWS) require careful consideration of label semantics
- The variable text quality (missing descriptions, short headlines) tests model robustness

**Learning Opportunities:** Working with this dataset will allow us to:
- Explore different text representation techniques (TF-IDF, word embeddings, transformer encodings)
- Practice handling class imbalance through various mitigation strategies
- Learn to evaluate multi-class classification with appropriate metrics (macro-F1, confusion matrices)
- Potentially experiment with pretrained language models for transfer learning

**Computational Efficiency:** Text data is generally less computationally demanding than high-resolution images, allowing for faster iteration during model development and hyperparameter tuning. This is particularly valuable given typical compute constraints in coursework settings.

**Personal Interest:** News categorization is a tangible, interpretable task where we can examine model predictions and understand why certain misclassifications occur, making the learning experience more engaging and insightful.

## Problem 2 – Frame the Problem (15 pts)

#### Objective

Identify the **key challenges** in your chosen dataset and outline **practical solutions** you would try, plus how you’ll **evaluate** them later.

#### Steps to follow

1. **Diagnose likely challenges (from your EDA):**

   Examples:
   * **Class imbalance:**
     Report label counts and an imbalance ratio (max / median). List any minority classes.
   * **Length/size variance:**
     For text, show length percentiles (50/75/90/95) and estimate truncation rate at candidate `max_text_length`s (e.g., 256/300/512). For images, summarize native resolutions.
   * **Noise/duplicates/leakage:**
     Note empty or malformed items, near-duplicates, and how you would prevent cross-split leakage.
   * **Ambiguous/overlapping labels:**
     Give 2–3 example pairs you expect to be confusable and why.
   * **Compute constraints:**
     Briefly state limits (RAM/GPU/CPU) that might affect batch size, sequence length, or image size.

2. **Map each challenge to a concrete solution plan:**

   Examples:
   * **Imbalance →** `class_weight` or oversampling; report which one you’d try first and why.
   * **Length/size →** pick a target `max_text_length` (e.g., 95th percentile) with masking; for images, standardize resize/crop and basic augmentation.
   * **Noise/duplicates →** dedupe (hash/near-dup), drop empty/very short items, document any relabeling.
   * **Ambiguity →** consider merging labels (if justified), or add features (bigrams/char-ngrams; simple image augmentations).
   * **Overfitting risk →** early stopping on your primary metric, dropout/weight decay, freeze-then-finetune plan (for pretrained features).

3. **Explore appropriate evaluation metrics:**

   Examples:
   * **Primary metric:** pick one aligned to your data (e.g., **macro-F1** if imbalanced; accuracy if balanced).
   * **Secondary metric(s):** per-class precision/recall, confusion matrix.
   * **Protocol:** stratified Train/Val/Test (e.g., 70/15/15), fixed seed, leakage checks.

4. **Answer the graded questions below.**



In [ ]:
# ============================================
# Problem 2: Challenge Analysis & Solution Planning
# ============================================

print("=" * 60)
print("PROBLEM 2: DETAILED CHALLENGE ANALYSIS")
print("=" * 60)

# --- 1. Class Imbalance Analysis ---
print("\n--- CLASS IMBALANCE ANALYSIS ---")
print(f"Number of classes: {len(unique_categories)}")
print(f"Max samples (largest class): {max_count:,}")
print(f"Min samples (smallest class): {min_count:,}")
print(f"Median samples: {median_count:,.0f}")
print(f"Imbalance ratio (max/min): {max_count/min_count:.2f}x")
print(f"Imbalance ratio (max/median): {max_count/median_count:.2f}x")

# Identify minority classes (less than 25% of median)
minority_threshold = median_count * 0.5
minority_classes = category_df[category_df['Count'] < minority_threshold]
print(f"\nMinority classes (< 50% of median = {minority_threshold:.0f} samples):")
print(minority_classes.to_string(index=False))

# --- 2. Text Length Analysis for Truncation Planning ---
print("\n--- TEXT LENGTH PERCENTILES (for max_length selection) ---")
print("\nCombined text word count percentiles:")
for p in [50, 75, 90, 95, 99]:
    val = np.percentile(combined_word_counts, p)
    print(f"  {p}th percentile: {val:.0f} words")

# Estimate truncation rates at different max_length values (in tokens, roughly 1.3x words)
print("\nEstimated truncation rates at different max_length values:")
for max_len in [64, 128, 256, 512]:
    # Rough approximation: 1 word ≈ 1.3 tokens for BERT-style tokenizers
    word_equiv = max_len / 1.3
    truncated = sum(1 for w in combined_word_counts if w > word_equiv)
    pct = 100 * truncated / len(combined_word_counts)
    print(f"  max_length={max_len}: ~{pct:.1f}% texts would be truncated")

# --- 3. Empty/Noise Analysis ---
print("\n--- NOISE & DATA QUALITY ANALYSIS ---")
print(f"Empty headlines: {empty_headline} ({100*empty_headline/len(huff_all):.2f}%)")
print(f"Empty descriptions: {empty_description} ({100*empty_description/len(huff_all):.2f}%)")

# Very short texts (< 5 words combined)
very_short = sum(1 for w in combined_word_counts if w < 5)
print(f"Very short texts (<5 words): {very_short} ({100*very_short/len(huff_all):.2f}%)")

# --- 4. Duplicate Analysis for Leakage Prevention ---
print("\n--- DUPLICATE ANALYSIS (for leakage prevention) ---")
print(f"Unique headlines: {len(headline_counter):,}")
print(f"Duplicate headline entries: {duplicate_headlines:,}")
print(f"Unique combined texts: {len(combined_counter):,}")
print(f"Duplicate combined text entries: {duplicate_combined:,}")

# --- 5. Category Overlap Analysis ---
print("\n--- POTENTIALLY OVERLAPPING CATEGORIES ---")
overlap_candidates = [
    ("PARENTING", "PARENTS", "Nearly identical topics"),
    ("HEALTHY LIVING", "WELLNESS", "Synonymous health topics"),
    ("ARTS", "CULTURE & ARTS", "Arts subset"),
    ("STYLE", "STYLE & BEAUTY", "Style subset"),
    ("WORLDPOST", "THE WORLDPOST", "Same publication/topic"),
    ("TASTE", "FOOD & DRINK", "Both culinary content"),
    ("POLITICS", "WORLD NEWS", "Political content overlap"),
]

print("\nCategory pairs that may benefit from merging:")
for cat1, cat2, reason in overlap_candidates:
    if cat1 in unique_categories and cat2 in unique_categories:
        c1 = category_counts.get(cat1, 0)
        c2 = category_counts.get(cat2, 0)
        print(f"  {cat1} ({c1:,}) + {cat2} ({c2:,}) = {c1+c2:,} → Reason: {reason}")

# --- 6. Proposed Train/Val/Test Split ---
print("\n--- PROPOSED DATA SPLIT ---")
total = len(huff_all)
train_pct, val_pct, test_pct = 0.80, 0.10, 0.10
print(f"Proposed stratified split: {int(train_pct*100)}/{int(val_pct*100)}/{int(test_pct*100)}")
print(f"  Train: ~{int(total*train_pct):,} samples")
print(f"  Val:   ~{int(total*val_pct):,} samples")
print(f"  Test:  ~{int(total*test_pct):,} samples")
print("Using stratified sampling to maintain class proportions in each split.")

### Graded Questions (3 pts each)

For each question, answer thoroughly but concisely, in a short paragraph, longer or shorter as needed. Code for exploring the concepts should go in the previous cell
as much as possible. 

1. **State the prediction task**  
   - Describe what your model will predict (the *label*).  
   - *Examples:*  
     - “Given a photo of food, predict which of 101 categories it belongs to.”  
     - “Given a news headline + summary, predict its topical category.”  

2.1. **Prediction Task:**

Given a news article's **headline combined with its short description** (formatted as `"headline [SEP] short_description"`), our model will predict which of **41 topical categories** the article belongs to. This is a **multi-class text classification** task.

For example:
- Input: `"Trump Will Nominate 'Torture Memo' Lawyer To Transportation Post [SEP] Steven G. Bradbury wrote the Bush administration's legal justification..."` 
- Output: `POLITICS`

The model learns to recognize linguistic patterns, keywords, and semantic features in headlines and descriptions that are indicative of specific news categories.

2. **Define inputs and outputs**  
   - *Inputs:* what information the model receives (e.g., pixel data, tokenized text).  
   - *Outputs:* the categorical label the model will predict.  

2.2. **Inputs and Outputs:**

**Inputs:**
- **Raw form:** A concatenated text string consisting of the news headline and short description, separated by `[SEP]` token: `"headline [SEP] short_description"`
- **Processed form (for deep learning):** Tokenized text converted to integer sequences using a tokenizer (e.g., DistilBERT tokenizer or Keras TextVectorization). This includes:
  - `input_ids`: Integer token IDs representing the text
  - `attention_mask`: Binary mask indicating valid tokens vs. padding
  - Sequence length: We plan to use `max_length=128` tokens (covering ~95% of texts without truncation)
- **Alternative (for classical ML):** TF-IDF vectors or word embedding averages (e.g., 300-dimensional GloVe embeddings)

**Outputs:**
- **Model output:** A probability distribution over 41 classes (softmax output)
- **Prediction:** The category with the highest probability
- **Label format:** Integer class ID (0-40) corresponding to one of the 41 news categories
- **Categories include:** POLITICS, ENTERTAINMENT, WELLNESS, TRAVEL, PARENTING, FOOD & DRINK, BUSINESS, SPORTS, COMEDY, CRIME, and 31 others

3. **Identify possible challenges**  
   - Imbalanced classes, noisy data, ambiguous labels, overlapping features, or missing data  
   - *Images:* variation in lighting, color, composition, or size.  
   - *Text:* class imbalance, duplicate stories, short or ambiguous headlines.  

2.3. **Identified Challenges:**

1. **Significant Class Imbalance:**
   - The imbalance ratio (max/min) is substantial, with POLITICS having many times more samples than minority classes
   - Minority classes like EDUCATION and COLLEGE have far fewer samples than the median
   - Risk: Model may achieve high accuracy by favoring majority classes while performing poorly on minority classes

2. **Overlapping/Ambiguous Categories:**
   - Several category pairs are semantically near-identical: PARENTING vs. PARENTS, HEALTHY LIVING vs. WELLNESS, ARTS vs. CULTURE & ARTS
   - Some pairs have significant topical overlap: POLITICS vs. WORLD NEWS (international political stories)
   - Risk: Systematic confusion between these pairs will limit achievable accuracy

3. **Missing/Empty Data:**
   - A notable percentage of samples have empty short descriptions
   - Some texts are very short (<5 words), providing limited signal for classification
   - Risk: Model may struggle with headline-only samples that lack descriptive context

4. **Duplicate Content:**
   - Duplicate headlines and combined texts exist in the dataset
   - Risk: Data leakage if duplicates appear in both train and test sets

5. **Compute Constraints:**
   - With ~200K samples and 41 classes, training transformer models may require significant GPU memory
   - Batch size and sequence length choices will be constrained by available resources

4. **Propose how you will prepare or improve the data to address the challenges**  
   - *Images:* resizing, normalization, data augmentation (flips, rotations, brightness, color jitter).  
   - *Text:* tokenization, stop-word removal, TF-IDF, class balancing, embeddings (choose an embedding approach and specify its vector size). 

2.4. **Data Preparation & Improvement Plan:**

**Addressing Class Imbalance:**
- **Primary approach:** Use `class_weight='balanced'` or compute custom class weights inversely proportional to class frequency
- **Alternative:** Oversample minority classes using techniques like random oversampling or SMOTE for text
- **Rationale:** Class weighting is simpler to implement and doesn't increase dataset size, making it our first choice

**Addressing Length Variance:**
- Set `max_length=128` tokens (covers ~95% of texts based on our percentile analysis)
- Use dynamic padding within batches to reduce computation waste
- For very short texts, padding will ensure consistent input dimensions

**Addressing Missing/Noisy Data:**
- Remove samples with empty headlines (no useful signal)
- Keep samples with empty descriptions (headline-only classification is still meaningful)
- Drop exact duplicates before train/test split to prevent leakage
- Minimum text length filter: Consider removing texts with <3 words total

**Addressing Ambiguous Categories:**
- **Option A (Conservative):** Keep all 41 categories and accept some inter-class confusion
- **Option B (Recommended):** Merge clearly synonymous pairs:
  - PARENTING + PARENTS → PARENTING
  - HEALTHY LIVING + WELLNESS → WELLNESS
  - This reduces classes and eliminates artificial confusion

**Text Preprocessing Pipeline:**
- Use pretrained transformer tokenizer (DistilBERT) for subword tokenization
- Alternatively, for classical baseline: spaCy for lemmatization + TF-IDF vectorization
- Embedding approach: DistilBERT embeddings (768 dimensions) or GloVe (300 dimensions)

**Overfitting Prevention:**
- Early stopping based on validation macro-F1 (patience=5 epochs)
- Dropout (0.3-0.5) in classification head
- Weight decay (L2 regularization) with AdamW optimizer
- Freeze pretrained layers initially, then fine-tune top layers

5. **Specify success metrics**  
   - Identify the metrics you plan to use to evaluate model performance—typically **accuracy** and/or **F1-score**, which are standard for classification tasks.  
   - Briefly explain **why** these metrics are appropriate for your dataset and goal. For instance, accuracy may suffice for well-balanced datasets, while F1-score better reflects performance when some classes are under-represented.
   - If your dataset is **imbalanced**, consider computing **per-class metrics** (e.g., precision, recall, or F1 for each label) or **macro-averaged** scores, which give equal weight to each class regardless of its size—ensuring that minority classes are evaluated fairly.
In some cases, weighted averages (which weight classes by their frequency) or **confusion matrices** can also provide useful insight.
> You haven't run any models yet, and we haven’t studied every possible metric, but you’re encouraged to ask your favorite generative AI tool which evaluation metrics might best fit your dataset!
   - Clearly state how you will interpret success—for example, “Our goal is to achieve at least 80% overall accuracy without large per-class disparities.”

2.5. **Success Metrics:**

**Primary Metric: Macro-F1 Score**
- **Why:** Given the significant class imbalance, macro-F1 is more appropriate than accuracy because it gives equal weight to each class regardless of size. This ensures minority classes are evaluated fairly and prevents the model from achieving high scores by simply predicting majority classes.
- **Calculation:** Average of F1 scores computed independently for each of the 41 classes

**Secondary Metrics:**
- **Overall Accuracy:** Still useful as a simple, interpretable measure, but should be interpreted alongside macro-F1
- **Weighted F1:** Accounts for class imbalance by weighting each class's F1 by its support; useful for understanding real-world performance distribution
- **Per-Class Precision & Recall:** Identifies which specific categories the model struggles with
- **Confusion Matrix:** Visualizes inter-class confusion patterns, especially useful for identifying systematic errors between overlapping categories (e.g., POLITICS ↔ WORLD NEWS)

**Evaluation Protocol:**
- **Split:** Stratified 80/10/10 train/validation/test split with fixed random seed (42) for reproducibility
- **Leakage prevention:** Deduplicate before splitting; no data shared between splits
- **Validation:** Use validation set for hyperparameter tuning and early stopping
- **Final evaluation:** Report test set metrics only once, after all tuning is complete

**Success Criteria:**
- **Goal:** Achieve at least **60% macro-F1** on the test set (reasonable for 41-class problem with imbalance and ambiguity)
- **Stretch goal:** **70% macro-F1** with careful category merging and class balancing
- **Red flag:** If accuracy is high (>80%) but macro-F1 is low (<50%), the model is likely biased toward majority classes and needs rebalancing

### Final Question: Describe what use you made of generative AI tools in preparing this Milestone. 

**AI Question:**

In preparing this milestone, we made extensive use of **Claude Code** (an AI assistant) to help with the following tasks:

1. **EDA Code Generation:** Claude Code helped write comprehensive exploratory data analysis code including class distribution visualization, text length analysis, duplicate detection, and identification of potentially overlapping categories.

2. **Analysis Structure:** The AI assistant helped organize the EDA into logical sections (basic statistics, class distribution, missing data, text lengths, duplicates, category overlap) for clarity and completeness.

3. **Answer Formulation:** Claude Code assisted in drafting thorough responses to the graded questions, ensuring they addressed all required points while remaining concise and well-structured.

4. **Best Practices:** The AI provided guidance on appropriate preprocessing strategies (class weighting, max_length selection based on percentiles, deduplication for leakage prevention) and evaluation metrics (macro-F1 for imbalanced multi-class problems).

5. **Code Quality:** Claude Code helped ensure the analysis code was well-commented, reproducible (with fixed random seeds), and followed good practices for data science workflows.

All AI-generated content was reviewed for accuracy and relevance to our specific dataset and project requirements. The AI served as a productivity tool and knowledge resource, while the team made final decisions about dataset selection, challenge prioritization, and solution approaches.

---

## Appendix: A quick guide to Hugging Face Datasets

#### 1) What are they?

* A **table-like** dataset: rows = examples, columns = named fields (e.g., `"image"`, `"label"`, `"headline"`).
* Backed by **Apache Arrow** → fast, memory-efficient, lazy transforms.
* Two core objects:

  * `Dataset` — one table of rows/columns.
  * `DatasetDict` — a dict of splits, e.g. `{"train": Dataset, "val": Dataset, "test": Dataset}`.


#### 2) Load and inspect

```python
from datasets import load_dataset

# Food-101 (images)
food = load_dataset("food101", split="train+validation")  # both splits at once
len(food), food.column_names, food.features
# -> (≈101000, ['image','label'], {'label': ClassLabel(num_classes=101, names=[...])})

# Access by name (not by numeric column index!)
row0 = food[0]
img0, y0 = row0["image"], row0["label"]     # PIL image, int id
label_names = food.features["label"].names
label_names[y0]
```

For text (HuffPost JSON mirror):

```python
url = "https://huggingface.co/datasets/khalidalt/HuffPost/resolve/main/News_Category_Dataset_v2.json"
huff = load_dataset("json", data_files=url, split="train")
huff.column_names  # e.g. ['headline','short_description','category','authors','link','date']
```


#### 3) Common transforms

`Dataset`s are **immutable**: ops return a new dataset.

* **Map** (add/modify columns):

```python
def mk_text(ex):
    h = (ex.get("headline") or "").strip()
    s = (ex.get("short_description") or "").strip()
    return {"text": (h + " [SEP] " + s).strip()}

huff = huff.map(mk_text)  # adds 'text' column
```

* **Class-encode** labels (strings → integers with a vocabulary):

```python
from datasets.features import ClassLabel
if not isinstance(huff.features["category"], ClassLabel):
    huff = huff.class_encode_column("category")  # now ints with .names
```

* **Filter / select / rename / drop**:

```python
small = huff.select(range(5000))  # first 5k rows
huff = huff.remove_columns(["authors","link","date"])
huff = huff.rename_column("category", "label")
```


#### 4) Splitting & shuffling

```python
# Stratified 80/10/10 on Food-101 by 'label'
from datasets import DatasetDict
label_col = "label"

tmp = food.train_test_split(test_size=0.10, seed=42, stratify_by_column=label_col)
train_val = tmp["train"].train_test_split(test_size=1/9, seed=42, stratify_by_column=label_col)
ds = DatasetDict(train=train_val["train"], val=train_val["test"], test=tmp["test"])

len(ds["train"]), len(ds["val"]), len(ds["test"])
```

* `train_test_split` is **random by default** (reproducible with `seed=`).
* You typically **don’t need to pre-shuffle** datasets if your training dataloader already shuffles each epoch.


#### 5) Working with images

Use `with_transform` to apply on-the-fly resizing/augmentation and return tensors:

```python
import torchvision.transforms as T
from torch.utils.data import DataLoader
import torch, math, random

IM_SIZE = 224
train_tfms = T.Compose([T.RandomResizedCrop(IM_SIZE), T.RandomHorizontalFlip(), T.ToTensor()])
eval_tfms  = T.Compose([T.Resize(256), T.CenterCrop(IM_SIZE), T.ToTensor()])

def add_pixel_values(ex, tfms):  # ex['image'] -> ex['pixel_values']
    ex = dict(ex); ex["pixel_values"] = tfms(ex["image"]); return ex

train_t = ds["train"].with_transform(lambda ex: add_pixel_values(ex, train_tfms))
val_t   = ds["val"].with_transform(lambda ex: add_pixel_values(ex, eval_tfms))

def collate(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
            "labels": torch.tensor([b["label"] for b in batch])}

train_loader = DataLoader(train_t, batch_size=64, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(val_t,   batch_size=64, shuffle=False, collate_fn=collate)
```

### 6) Working with Text

You can preprocess and tokenize text datasets using either a **transformer tokenizer** or a **linguistic pipeline like spaCy**, depending on your model type and goals.


#### Option A: Transformer Tokenizer (for fine-tuning models like BERT or DistilBERT)

Use a pretrained tokenizer with the Hugging Face `map` method to efficiently process your dataset in batches:

```python
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tok(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized = huff.map(
    tokenize,
    batched=True,
    remove_columns=[c for c in huff.column_names if c not in ("category", "label")]
)
```

This produces token IDs, attention masks, and other fields expected by transformer models.
Use this approach if your project involves fine-tuning pretrained language models.


#### Option B: spaCy Tokenization and Cleaning (for classical ML or custom preprocessing)

If you are **not using transformers**, or if you want to explore feature engineering (e.g., TF-IDF, word frequency, or embedding averages), you can preprocess text with **spaCy** instead.

```python
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy
from datasets import load_dataset

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "textcat"])
STOP = spacy.lang.en.stop_words.STOP_WORDS

def spacy_clean(batch):
    docs = list(nlp.pipe(batch["text"], batch_size=1000))
    cleaned = []
    for doc in docs:
        tokens = [t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop]
        cleaned.append(" ".join(tokens))
    return {"text_clean": cleaned, "len_tokens": [len(c.split()) for c in cleaned]}

huff = huff.map(
    spacy_clean,
    batched=True,
    batch_size=1000,
    remove_columns=[c for c in huff.column_names if c not in ("category", "label")]
)
```

The resulting column `text_clean` can be used with:

* `TfidfVectorizer` (Scikit-learn)
* `TextVectorization` (Keras)
* or any other custom embedding method.

> 💡 **Tip:** spaCy is ideal for lightweight NLP pipelines or for models that rely on explicit preprocessing (lemmatization, stopword removal). Transformer tokenizers, by contrast, expect *raw text* and handle subword tokenization internally.



#### 7) Slicing, concatenating, saving

```python
# Slicing
head_1000 = food.select(range(1000))
tail_10pct = food.select(range(int(0.9*len(food)), len(food)))

# Concatenate splits/datasets
from datasets import concatenate_datasets
all_train = concatenate_datasets([ds["train"], ds["val"]])

# Save / reload
ds.save_to_disk("food101_splits")
from datasets import load_from_disk
ds2 = load_from_disk("food101_splits")
```


#### 8) Quick “gotchas”

* **Columns by name** (strings), not numeric indices.
* Avoid converting huge columns to `list(...)` unless necessary; prefer vectorized ops with `map`, `filter`, `select`.
* `PYTHONHASHSEED` must be set **before** the Python process starts to matter; use explicit `seed=` arguments for reproducibility.
* Datasets print **previews** (e.g., `Column([6, 6, 6, ...])` is just the first few values).


#### 9) A minimal checklist to follow

1. `load_dataset(...)` → confirm `column_names`, `features`.
2. Build any needed columns (`"text"`), and **class-encode** labels if strings.
3. Make a **stratified 80/10/10** split (`train_test_split` ×2).
4. Do **EDA**: class counts, sample printouts or image grids.
5. For training:

   * **images** → `with_transform` + DataLoader
   * **text** → tokenizer via `.map(...)` + trainer/model pipeline
6. Save your `DatasetDict` with `save_to_disk(...)` (optional for text but handy, **don't** use it for big image datasets).

